In [1]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. 텍스트 데이터 준비
data = (
    "RNN models are used for sequential data prediction. "
    "They are useful in natural language processing tasks. "
    "Text generation is one popular example of using RNNs."
)

# 2. 토크나이저 설정 및 텍스트 정수 인코딩
tokenizer = Tokenizer(
    filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~',  # 제거할 특수문자 설정
    lower=True,                                 # 모두 소문자로 변환
    oov_token="<OOV>"                           # 단어장에 없는 단어는 <OOV>로 처리
)
tokenizer.fit_on_texts([data])                  # 텍스트 데이터에 대한 단어장 생성
encoded = tokenizer.texts_to_sequences([data])[0]  # 전체 텍스트를 정수 시퀀스로 변환

# 3. 입력 시퀀스 생성
sequences = []
for i in range(1, len(encoded)):
    sequence = encoded[:i+1]    # 점점 늘어나는 시퀀스 추출
    sequences.append(sequence) # 예: [1, 2], [1, 2, 3], ...

# 4. 시퀀스 길이 통일 (모두 가장 긴 길이에 맞춰 앞을 0으로 채움)
max_len = max(len(seq) for seq in sequences)  # 가장 긴 시퀀스 길이 계산
sequences = pad_sequences(sequences, maxlen=max_len, padding='pre')  # 앞쪽 0 패딩

# 5. 입력(X)과 출력(y) 분리
sequences = np.array(sequences)
X = sequences[:, :-1]  # 마지막 단어를 제외한 부분이 입력
y = sequences[:, -1]   # 마지막 단어가 정답 (예측 대상)

# 출력 y를 One-hot encoding → [0 0 1 0 ...] 형식으로 변환
vocab_size = len(tokenizer.word_index) + 1  # 전체 어휘 수 (패딩 포함)
y = to_categorical(y, num_classes=vocab_size)

# 6. RNN 모델 구성
model = Sequential()
model.add(Input(shape=(max_len-1,)))  # 입력 시퀀스 길이 지정
# Embedding 층(input_dim: 전체 단어 수, output_dim: 64차원 벡터로 임베딩)
model.add(Embedding(input_dim=vocab_size, output_dim=64))
# RNN 층(128개의 유닛을 가진 순환 신경망)
model.add(LSTM(128))
# 출력층(각 단어에 대한 확률 분포 예측)
model.add(Dense(vocab_size, activation='softmax'))

# 7. 모델 컴파일 및 학습
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=200, verbose=1)

# 8. 다음 단어 예측 함수
def predict_next_word(model, tokenizer, text, max_len=max_len):
    # 입력 텍스트를 정수 시퀀스로 변환
    encoded = tokenizer.texts_to_sequences([text])[0]
    # 길이 맞춰 앞쪽에 0 패딩
    encoded = pad_sequences([encoded], maxlen=max_len-1, padding='pre')
    # 예측 (다음 단어 확률 분포)
    y_pred = model.predict(encoded, verbose=0)
    predicted_index = np.argmax(y_pred)  # 가장 확률 높은 단어의 인덱스
    return tokenizer.index_word.get(predicted_index, "<UNK>")  # 인덱스를 단어로 변환

# Top-k 단어 예측 함수 (확률 상위 K개 반환)
def predict_top_k_words(model, tokenizer, text, k=3):
    encoded = tokenizer.texts_to_sequences([text])[0]
    encoded = pad_sequences([encoded], maxlen=max_len-1, padding='pre')
    y_pred = model.predict(encoded, verbose=0)[0]  # 예측 결과: 확률 벡터
    top_indices = y_pred.argsort()[-k:][::-1]      # 확률 상위 k개 인덱스
    return [(tokenizer.index_word.get(i, "<UNK>"), y_pred[i]) for i in top_indices]
    # 단어와 확률을 튜플로 반환

# 9. 테스트 예시
input_text = "RNN models are"
predicted_word = predict_next_word(model, tokenizer, input_text)
print(f"\n▶ Input: '{input_text}'")
print(f"→ Predicted next word: '{predicted_word}'")

# Top-3 단어 확인
top_words = predict_top_k_words(model, tokenizer, input_text, k=3)
print("\nTop 3 predicted words:")
for word, prob in top_words:
    print(f"  - {word} ({prob:.4f})")

Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.0000e+00 - loss: 3.2588
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.1667 - loss: 3.2512
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.2083 - loss: 3.2431
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.3333 - loss: 3.2339
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.3333 - loss: 3.2227
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.2500 - loss: 3.2080
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.2083 - loss: 3.1875
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.1667 - loss: 3.1574
Epoch 9/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.1250 - loss: 3.1132
Epoch 10/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.1250 - loss: 3.0592
Epoch 11/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.1250 - loss: 3.0322
Epoch 12/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step - accuracy: 0.0833 -